## Ensemble Engineer

In [1]:
import pandas as pd

# Cargamos datos
train_df = pd.read_csv('../data/processed/train_balanced.csv')
test_df = pd.read_csv('../data/processed/test_original.csv')

X_train = train_df.iloc[:, :-1]
y_train = train_df.iloc[:, -1]

X_test = test_df.iloc[:, :-1]
y_test = test_df.iloc[:, -1]

print(f"Éxito. Entrenando con {X_train.shape[1]} columnas.")
print(f"La columna objetivo detectada fue: '{train_df.columns[-1]}'")

Éxito. Entrenando con 44 columnas.
La columna objetivo detectada fue: 'y'


### 1. Implementar VotingClassifier (hard y soft voting)

- Carga de los Mejores Candidatos

In [2]:
import joblib
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report

# Cargamos los 4 modelos optimizados
tuned_rf = joblib.load('../models/tuned_rf.pkl')
tuned_knn = joblib.load('../models/tuned_knn.pkl')
tuned_lr = joblib.load('../models/tuned_lr.pkl')
tuned_dt = joblib.load('../models/tuned_dt.pkl')

# Definimos la lista de estimadores
estimators_list = [
    ('rf', tuned_rf),
    ('knn', tuned_knn),
    ('lr', tuned_lr),
    ('dt', tuned_dt)
]

print("Candidatos cargados y lista de estimadores preparada.")

Candidatos cargados y lista de estimadores preparada.


#### 1.1. Hard Voting

In [3]:
# Función para limpiar los prefijos de las columnas
def clean_column_names(df):
    # Quita 'num__' o 'cat__' del inicio de los nombres de las columnas
    df.columns = [col.replace('num__', '').replace('cat__', '').replace('remainder__', '') for col in df.columns]
    return df

# Limpiamos tanto el train como el test
X_train = clean_column_names(X_train)
X_test = clean_column_names(X_test)

print("Nuevas columnas en X_train:", X_train.columns.tolist()[:5])

Nuevas columnas en X_train: ['age', 'education', 'campaign', 'pdays', 'previous']


In [4]:
# Crear el modelo Hard Voting
voting_hard = VotingClassifier(
    estimators=estimators_list,
    voting='hard'
)

# Entrenar el ensamble (coordinar los modelos con los datos)
voting_hard.fit(X_train, y_train)

# Predicción y Evaluación
preds_hard = voting_hard.predict(X_test)
print("--- Reporte Hard Voting ---")
print(classification_report(y_test, preds_hard))

/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


--- Reporte Hard Voting ---
              precision    recall  f1-score   support

           0       0.93      0.94      0.94      7307
           1       0.50      0.46      0.48       927

    accuracy                           0.89      8234
   macro avg       0.72      0.70      0.71      8234
weighted avg       0.88      0.89      0.89      8234



#### 1.2. Soft Voting

In [5]:
# Implementar VotingClassifier (Soft Voting)
voting_soft = VotingClassifier(
    estimators=estimators_list,
    voting='soft'
)

# Entrenar (ya con las columnas limpias)
voting_soft.fit(X_train, y_train)

# Predicción y Evaluación
preds_soft = voting_soft.predict(X_test)
print("--- Reporte Soft Voting ---")
print(classification_report(y_test, preds_soft))

/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


--- Reporte Soft Voting ---
              precision    recall  f1-score   support

           0       0.94      0.92      0.93      7307
           1       0.46      0.54      0.49       927

    accuracy                           0.88      8234
   macro avg       0.70      0.73      0.71      8234
weighted avg       0.89      0.88      0.88      8234



El modelo de Soft Voting es seleccionado como el ganador debido a su mejor desempeño en la captura de la clase minoritaria (clientes que compran), logrando un Recall de 0.54 en comparación al 0.46 del Hard Voting. Al promediar las probabilidades de los modelos optimizados, este ensamble ofrece un equilibrio superior (Macro F1-Score de 0.71), minimizando el riesgo de sesgo. Esta estabilidad lo convierte en la herramienta más confiable para las campañas de marketing del banco, asegurando que identifiquemos a más clientes potenciales sin sacrificar drásticamente la precisión general.

### 2. Implementar BaggingClassifier (modelos con overfitting)

- Identificación del Overfitting

In [6]:
modelos_a_probar = {'Decision Tree': tuned_dt, 'Random Forest': tuned_rf}

for nombre, modelo in modelos_a_probar.items():
    train_acc = modelo.score(X_train, y_train)
    test_acc = modelo.score(X_test, y_test)
    print(f"{nombre} - Train: {train_acc:.4f} | Test: {test_acc:.4f} | Dif: {train_acc - test_acc:.4f}")

Decision Tree - Train: 0.6293 | Test: 0.6752 | Dif: -0.0460
Random Forest - Train: 0.7862 | Test: 0.7974 | Dif: -0.0112


#### 2.1. Implementación del BaggingClassifier

In [7]:
from sklearn.ensemble import BaggingClassifier

# Implementamos Bagging sobre el Random Forest que mostró mayor overfitting (Dif: 0.0862)
bagging_rf = BaggingClassifier(
    estimator=tuned_rf,
    n_estimators=10,      # Creamos 10 variaciones del modelo
    max_samples=0.7,      # Cada una entrena con el 70% de los datos
    max_features=0.7,     # Cada una usa el 70% de las variables
    random_state=42,
    n_jobs=-1
)

# Entrenar el Bagging
print("Entrenando Bagging sobre Random Forest...")
bagging_rf.fit(X_train, y_train)

# Predicción y Evaluación
preds_bagging = bagging_rf.predict(X_test)
print("\n--- Reporte Bagging (Random Forest) ---")
print(classification_report(y_test, preds_bagging))

Entrenando Bagging sobre Random Forest...

--- Reporte Bagging (Random Forest) ---
              precision    recall  f1-score   support

           0       0.94      0.93      0.94      7307
           1       0.49      0.53      0.51       927

    accuracy                           0.89      8234
   macro avg       0.72      0.73      0.72      8234
weighted avg       0.89      0.89      0.89      8234



Se implementó un BaggingClassifier sobre el Random Forest para mejorar la estabilidad y generalización del modelo. Tras comparar los resultados, el Bagging se selecciona como el modelo ganador para los objetivos de captación comercial. Aunque los ensambles de Voting ofrecen un equilibrio conservador, el Bagging logra el mejor balance para la clase objetivo con un F1-Score (Macro) de 0.72 y un Recall de 0.53. Esta capacidad para identificar correctamente a los clientes potenciales, manteniendo una precisión sólida, lo convierte en la solución más robusta para maximizar el alcance de las campañas de marketing del banco sin saturar innecesariamente los canales de contacto.

### 3. Entrenar modelos de Gradient Boosting avanzado

#### 3.1. Implementación de XGBoost

In [8]:
!pip install xgboost lightgbm

In [9]:
from xgboost import XGBClassifier

# Configuración básica de XGBoost
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Entrenamiento
print("Entrenando XGBoost...")
xgb_model.fit(X_train, y_train)

# Evaluación
preds_xgb = xgb_model.predict(X_test)
print("\n--- Reporte XGBoost ---")
print(classification_report(y_test, preds_xgb))

Entrenando XGBoost...


/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [19:38:03] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- Reporte XGBoost ---
              precision    recall  f1-score   support

           0       0.93      0.95      0.94      7307
           1       0.53      0.41      0.47       927

    accuracy                           0.89      8234
   macro avg       0.73      0.68      0.70      8234
weighted avg       0.88      0.89      0.89      8234



#### 3.2. Implementación de LightGBM

In [10]:
from lightgbm import LGBMClassifier

# Configuración básica de LightGBM
lgbm_model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    verbose=-1
)

# Entrenamiento
print("Entrenando LightGBM...")
lgbm_model.fit(X_train, y_train)

# Evaluación
preds_lgbm = lgbm_model.predict(X_test)
print("\n--- Reporte LightGBM ---")
print(classification_report(y_test, preds_lgbm))

Entrenando LightGBM...

--- Reporte LightGBM ---
              precision    recall  f1-score   support

           0       0.93      0.95      0.94      7307
           1       0.52      0.40      0.45       927

    accuracy                           0.89      8234
   macro avg       0.72      0.67      0.69      8234
weighted avg       0.88      0.89      0.88      8234



Se evaluaron algoritmos de Gradient Boosting avanzado (XGBoost y LightGBM). En este escenario con datos no vistos, ambos modelos mostraron un desempeño inferior al Bagging, con un Recall de 0.41 y 0.40 respectivamente. Aunque mantienen un Accuracy general del 89%, su capacidad para detectar clientes potenciales (clase 1) es limitada en comparación con otras técnicas de ensamble. Por lo tanto, el Bagging Classifier sobre Random Forest se confirma como la solución óptima, ya que ofrece el mejor equilibrio y robustez para el negocio, superándolos con un Macro F1-Score de 0.72.

### 4.	Construir un StackingClassifier

In [11]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

# 1. Definimos los modelos base (los que ya tenemos cargados)
base_models = [
    ('rf', tuned_rf),
    ('knn', tuned_knn),
    ('xgb', xgb_model), 
    ('lgbm', lgbm_model)
]

# 2. Definimos el meta-modelo (el que toma la decisión final)
meta_model = LogisticRegression()

# 3. Creamos el Stacking
stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5, # Usa Cross-Validation para que el meta-modelo no se sobreajuste
    n_jobs=-1
)

# 4. Entrenamiento
print("Entrenando el Stacking Classifier...")
stacking_model.fit(X_train, y_train)

# 5. Evaluación
preds_stacking = stacking_model.predict(X_test)
print("\n--- Reporte Stacking Classifier ---")
print(classification_report(y_test, preds_stacking))

Entrenando el Stacking Classifier...


/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [19:39:57] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[19:40:09] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[19:40:09] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[19:40:09] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[19:40:09] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[19:40:09] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.




--- Reporte Stacking Classifier ---
              precision    recall  f1-score   support

           0       0.93      0.94      0.94      7307
           1       0.49      0.48      0.49       927

    accuracy                           0.88      8234
   macro avg       0.71      0.71      0.71      8234
weighted avg       0.88      0.88      0.88      8234



Se construyó un StackingClassifier integrando Random Forest, KNN, XGBoost y LightGBM, utilizando una Regresión Logística como meta-learner. Esta arquitectura logró un desempeño sólido con un Macro F1-Score de 0.71 y un Accuracy global de 0.88, demostrando que la combinación de modelos base permite capturar patrones complejos. No obstante, tras la comparativa final, el Bagging Classifier sobre Random Forest se mantiene como la solución preferida, ya que ofrece un equilibrio superior en la detección de clientes potenciales (F1-Score: 0.72) y una mayor exactitud general.

### 5. Evaluar todos los ensambles con cross-validation y comparar con los baselines tuneados.

In [14]:
from sklearn.model_selection import cross_validate

# Definimos las métricas que queremos ver
metrics = ['accuracy', 'precision', 'recall', 'f1']

modelos_finales = {
    'Tuned DT': tuned_dt,
    'Tuned RF': tuned_rf,
    'Tuned KNN': tuned_knn,
    'Tuned Logistic Reg': tuned_lr,
    'Hard Voting': voting_hard,
    'Soft Voting': voting_soft,
    'Bagging (RF)': bagging_rf,
    'XGBoost': xgb_model,
    'LightGBM': lgbm_model,
    'Stacking': stacking_model
}

model_results = []

print("Calculando métricas integrales (Accuracy, Precision, Recall, F1)...")

for nombre, modelo in modelos_finales.items():
    print(f"Evaluando {nombre}...")
    # Calculamos todas las métricas a la vez
    cv_results = cross_validate(modelo, X_train, y_train, cv=5, scoring=metrics, n_jobs=-1)
    
    # Guardamos los promedios de cada métrica
    model_results.append({
        'Modelo': nombre,
        'Accuracy': cv_results['test_accuracy'].mean(),
        'Precision': cv_results['test_precision'].mean(),
        'Recall': cv_results['test_recall'].mean(),
        'F1-Score': cv_results['test_f1'].mean()
    })
    print(f"{nombre} completado.")

# Creamos la tabla definitiva
df_model_performance = pd.DataFrame(model_results)
df_model_performance = df_model_performance.sort_values(by='F1-Score', ascending=False)

print("\n--- REPORTE DE DESEMPEÑO INTEGRAL ---")
print(df_model_performance)

Calculando métricas integrales (Accuracy, Precision, Recall, F1)...
Evaluando Tuned DT...
Tuned DT completado.
Evaluando Tuned RF...
Tuned RF completado.
Evaluando Tuned KNN...
Tuned KNN completado.
Evaluando Tuned Logistic Reg...


/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penal

Tuned Logistic Reg completado.
Evaluando Hard Voting...


/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent valu

Hard Voting completado.
Evaluando Soft Voting...


...

Soft Voting completado.
Evaluando Bagging (RF)...
Bagging (RF) completado.
Evaluando XGBoost...


/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [19:51:16] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [19:51:16] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [19:51:16] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [19:51:17] WARNING: /Users/runner/work/

XGBoost completado.
Evaluando LightGBM...
LightGBM completado.
Evaluando Stacking...


[19:51:21] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[19:51:21] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[19:51:21] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[19:51:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[19:51:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[19:51:38] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[19:51:39] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[19:51:39] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" }

Stacking completado.

--- REPORTE DE DESEMPEÑO INTEGRAL ---
               Modelo  Accuracy  Precision    Recall  F1-Score
5         Soft Voting  0.923568   0.915581  0.933081  0.921806
1            Tuned RF  0.927058   0.939703  0.912828  0.921121
6        Bagging (RF)  0.917649   0.927864  0.905608  0.912706
9            Stacking  0.919720   0.933851  0.905986  0.909356
4         Hard Voting  0.908565   0.934490  0.878819  0.901831
7             XGBoost  0.914913   0.950377  0.876529  0.899933
8            LightGBM  0.915598   0.951611  0.877042  0.899470
0            Tuned DT  0.898216   0.928195  0.863560  0.887637
2           Tuned KNN  0.873734   0.807117  0.982107  0.886033
3  Tuned Logistic Reg  0.742849   0.809709  0.634906  0.711694


#### Conclusión:

Tras evaluar las 10 arquitecturas mediante validación cruzada sobre el set de entrenamiento balanceado, se observa que el ensamble Soft Voting lidera el ranking teórico con un F1-Score de 0.9218.

- Capacidad de Aprendizaje (Recall: 93.3%): En condiciones controladas de laboratorio, este modelo demuestra la mayor sensibilidad para identificar perfiles de clientes potenciales.

- Precisión en Entrenamiento (91.5%): El modelo logra una alta tasa de acierto sobre la clase positiva dentro del entorno equilibrado.

#### Elección del Modelo:
Si bien el Soft Voting destaca en las métricas de entrenamiento, la evaluación definitiva realizada contra el set de prueba original (realidad del mercado) confirma que el Bagging sobre Random Forest es la solución más robusta y confiable. A diferencia de los modelos que tienden al sobreajuste en entornos ideales, el Bagging mantiene un mejor equilibrio y estabilidad ante el desbalance real de la base de datos. Por lo tanto, se recomienda su implementación final para optimizar las campañas de telemarketing, asegurando una captura eficiente de clientes sin comprometer la capacidad operativa por exceso de llamadas fallidas.